In [7]:
# Step 2 — single symbol-day sanity before any batch run.
# Pick a liquid, well-behaved shortlist name and a normal (non-Ramadan, non-Friday) day.
SYM  = "UBL"
DATE = "2026-06-30"

import sys
sys.path.insert(0, "existing_mm")   # adjust to wherever mm_backtest lives
# You'll need the parquet event loader from run_legacy_mm.py — import its build_events + run_one.
from run_legacy_mm import run_one, open_datasets   # if these names differ, grep and tell me

dsets = open_datasets(DATE)
result = run_one(DATE, SYM, dsets)   # returns the summary dict
import pprint; pprint.pprint(result)

# The three things that must be true before trusting any batch:
print("\nchecks:")
print("  eod fired:            ", result.get("eod_fired"))
print("  liquidation clean:    ", result.get("liquidation_clean"))
print("  fills > 0:            ", result.get("n_fills", 0) > 0)

{'date': '2026-06-30',
 'eod_fired': True,
 'equity_mid_mark': np.float64(-15833.453413634968),
 'halted_requotes': 0,
 'liquidation_clean': True,
 'n_cancels': 2469,
 'n_events': 18264,
 'n_fills': 583,
 'n_orders_sent': 12560,
 'net_pnl': np.float64(-16234.453378314967),
 'pos_at_close': 404.0,
 'rejected_crossing': 224,
 'rest_oid_resolved_frac': 0.0,
 'symbol': 'UBL',
 'unfilled_sh': 0.0}

checks:
  eod fired:             True
  liquidation clean:     True
  fills > 0:             True


In [2]:
import duckdb
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")
print(duckdb.sql(f"""
    SELECT resting_order_id, typeof(resting_order_id) AS type, count(*) n
    FROM read_parquet('{PARSED}/trades/date=2026-06-30/*.parquet')
    WHERE symbol='UBL' AND resting_order_id IS NOT NULL
    GROUP BY 1,2 ORDER BY n DESC LIMIT 5
""").df().to_string(index=False))

resting_order_id    type  n
0010THF0D00017T6 VARCHAR  8
0010THF0D00029AK VARCHAR  7
0010THF0D000IRHT VARCHAR  7
0010THF0D000JMPS VARCHAR  7
0010THF0D000L9UG VARCHAR  7


In [3]:
import duckdb
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")
print(duckdb.sql(f"""
    SELECT resting_order_id, typeof(resting_order_id) AS type, count(*) n
    FROM read_parquet('{PARSED}/trades/date=2026-06-30/*.parquet')
    WHERE symbol='UBL' AND resting_order_id IS NOT NULL
    GROUP BY 1,2 ORDER BY n DESC LIMIT 5
""").df().to_string(index=False))

resting_order_id    type  n
0010THF0D00017T6 VARCHAR  8
0010THF0D000L9UG VARCHAR  7
0010THF0D000IRHT VARCHAR  7
0010THF0D00029AK VARCHAR  7
0010THF0D000JMPS VARCHAR  7


In [5]:
import duckdb
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")
print("trades resting_order_id:")
print(duckdb.sql(f"SELECT DISTINCT resting_order_id FROM read_parquet('{PARSED}/trades/date=2026-06-30/*.parquet') WHERE symbol='UBL' LIMIT 5").df().to_string(index=False))
print("\nupdates order_id:")
print(duckdb.sql(f"SELECT DISTINCT order_id FROM read_parquet('{PARSED}/ob_updates/date=2026-06-30/*.parquet') WHERE symbol='UBL' LIMIT 5").df().to_string(index=False))

trades resting_order_id:
resting_order_id
0010THF0D0003Q14
0010THF0D0008UU8
0010THF0D0007NYD
0010THF0D00078X7
0010THF0D0009OZO

updates order_id:
        order_id
0010THF0D00000U0
0010THF0D00003EM
0010THF0D0000825
0010THF0D00009CS
0010THF0D0000DM7


In [8]:
import mm_backtest
print("FEE_TOTAL_PCT =", mm_backtest.FEE_TOTAL_PCT, "  (TREC target ~7.77e-05, retail ~1.77e-03)")
print("USE_TREC_FEE  =", getattr(mm_backtest, "USE_TREC_FEE", "NOT DEFINED"))

from run_legacy_mm import parse_rest_oid
print("parse_rest_oid bare string ->", parse_rest_oid("0010THF0D00017T6"), "(should echo the string, not None)")

FEE_TOTAL_PCT = 0.0017727   (TREC target ~7.77e-05, retail ~1.77e-03)
USE_TREC_FEE  = NOT DEFINED
parse_rest_oid bare string -> None (should echo the string, not None)


In [9]:
import duckdb
PARSED = ("/Users/shazzak/Library/CloudStorage/"
          "GoogleDrive-shazzak@gmail.com/My Drive/Capital Stake - Parsed")
print(duckdb.sql(f"""
    SELECT
      count(*) AS total,
      count(resting_order_id) AS has_rest_oid,
      round(100.0*count(resting_order_id)/count(*),1) AS pct_populated
    FROM read_parquet('{PARSED}/trades/date=2026-06-30/*.parquet')
    WHERE symbol='UBL'
""").df().to_string(index=False))

 total  has_rest_oid  pct_populated
  3352           625           18.6


In [10]:
import mm_backtest, run_legacy_mm
print(mm_backtest.__file__)
print(run_legacy_mm.__file__)

/Users/shazzak/PycharmProjects/HFT/existing_mm/mm_backtest.py
/Users/shazzak/PycharmProjects/HFT/existing_mm/run_legacy_mm.py


In [11]:
import mm_backtest
assert getattr(mm_backtest, "USE_TREC_FEE", None) is not None, "WRONG FILE — USE_TREC_FEE not defined"
assert abs(mm_backtest.FEE_TOTAL_PCT - 7.77e-05) < 1e-6, f"WRONG FEE: {mm_backtest.FEE_TOTAL_PCT}"
print("fixes confirmed live in", mm_backtest.__file__)
# ... then run the UBL backtest immediately after ...

AssertionError: WRONG FILE — USE_TREC_FEE not defined